# Shopify Blog Archive ID Checker

**Special case:** Blog archive pages (`/blogs/{slug}`) — bukan article, page, atau collection.

- ID diambil dari `var __st` dengan `"p": "blog"`
- CMS URL: `https://admin.shopify.com/store/{store}/content/blogs/{id}`
- Input: file `.txt` berisi daftar URL
- Output: `shopify_blog_ids.csv`

In [ ]:
from google.colab import files

# ── Input Store Name ─────────────────────────────────────────
STORE_NAME = input("Masukkan Shopify store name (contoh: sme-brother): ").strip()

if not STORE_NAME:
    raise ValueError("Store name tidak boleh kosong!")

ADMIN_BASE = f"https://admin.shopify.com/store/{STORE_NAME}"
print(f"\n Store name     : {STORE_NAME}")
print(f" Admin base URL : {ADMIN_BASE}/")

# ── Upload File URL ──────────────────────────────────────────
print("\n Silakan upload file .txt berisi URL blog archive Shopify...")
print("   Format: satu URL per baris, contoh:")
print("   https://smebrother.com/blogs/blog-name")
uploaded = files.upload()

if not uploaded:
    raise ValueError("Tidak ada file yang diupload!")

INPUT_FILE  = list(uploaded.keys())[0]
OUTPUT_FILE = "shopify_blog_ids.csv"

print(f"\n File diterima  : {INPUT_FILE}")
print(f" Output file    : {OUTPUT_FILE}")

Masukkan Shopify store name (contoh: sme-brother): sme-brother

 Store name     : sme-brother
 Admin base URL : https://admin.shopify.com/store/sme-brother/

 Silakan upload file .txt berisi URL blog archive Shopify...
   Format: satu URL per baris, contoh:
   https://smebrother.com/blogs/blog-name


Saving sisa.txt to sisa.txt

 File diterima  : sisa.txt
 Output file    : shopify_blog_ids.csv


In [ ]:
# ============================================================
# Shopify Blog Archive ID Checker
# Target   : /blogs/{slug} — halaman listing artikel per blog
# ID source: var __st -> { "p": "blog", "rid": <id> }
# CMS URL  : https://admin.shopify.com/store/{store}/content/blogs/{id}
# Output   : url, blog_slug, shopify_blog_id, cms_url, status
# Saves    : real-time ke CSV
# ============================================================

import re
import json
import time
import pathlib
import requests
import pandas as pd
from urllib.parse import urlparse, unquote
from datetime import datetime

# ── Config ──────────────────────────────────────────────────
DELAY_SECONDS   = 5
REQUEST_TIMEOUT = 30

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
)

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.8,id;q=0.7,zh;q=0.6",
    "Connection": "keep-alive",
})

# ── Logger ───────────────────────────────────────────────────
def log(level, msg):
    icons = {
        "INFO":  "i ",
        "OK":    "OK",
        "WARN":  "! ",
        "ERROR": "X ",
        "SKIP":  ">>",
        "SAVE":  "S ",
        "DONE":  "* ",
        "DELAY": "~ ",
        "LINK":  "@ ",
    }
    ts   = datetime.now().strftime("%H:%M:%S")
    icon = icons.get(level, "  ")
    print(f"[{ts}] {icon} [{level}] {msg}")

# ── Validation ───────────────────────────────────────────────
def is_blog_archive_url(url):
    try:
        path  = unquote(urlparse(url).path).strip("/")
        parts = path.split("/")
        return len(parts) == 2 and parts[0] == "blogs"
    except Exception:
        return False

# ── Slug Extractor ───────────────────────────────────────────
def extract_blog_slug(url):
    try:
        path  = unquote(urlparse(url).path)
        parts = path.strip("/").split("/")
        if len(parts) >= 2 and parts[0] == "blogs":
            return parts[1]
    except Exception:
        pass
    return None

# ── ID Extractor ─────────────────────────────────────────────
def extract_blog_id_from_html(html):
    # Metode 1: var __st
    m = re.search(r"var\s+__st\s*=\s*(\{.*?\});", html, flags=re.DOTALL)
    if m:
        try:
            payload = json.loads(m.group(1))
            if payload.get("p") == "blog":
                rid = payload.get("rid")
                if isinstance(rid, int):
                    return rid
                if isinstance(rid, str) and rid.isdigit():
                    return int(rid)
        except json.JSONDecodeError:
            pass

    # Metode 2: resourceId dari ShopifyAnalytics
    m2 = re.search(
        r'"resourceType"\s*:\s*"blog"[^}]*?"resourceId"\s*:\s*(\d+)',
        html, flags=re.DOTALL
    )
    if m2:
        return int(m2.group(1))

    # Metode 3: var meta
    m3 = re.search(
        r'var\s+meta\s*=\s*\{.*?"pageType"\s*:\s*"blog".*?"resourceId"\s*:\s*(\d+)',
        html, flags=re.DOTALL
    )
    if m3:
        return int(m3.group(1))

    return None

# ── CMS URL Builder ──────────────────────────────────────────
def build_cms_url(store_name, blog_id):
    return f"https://admin.shopify.com/store/{store_name}/content/blogs/{blog_id}"

# ── Main Fetcher ─────────────────────────────────────────────
def fetch_blog_id(url):
    result = {
        "url":             url,
        "blog_slug":       None,
        "shopify_blog_id": None,
        "cms_url":         None,
        "status":          None,
    }

    if not is_blog_archive_url(url):
        result["status"] = "skipped (bukan blog archive URL)"
        return result

    result["blog_slug"] = extract_blog_slug(url)

    try:
        resp = SESSION.get(url, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
    except requests.exceptions.HTTPError as e:
        result["status"] = f"error HTTP {e.response.status_code}"
        return result
    except Exception as e:
        result["status"] = f"error ({e})"
        return result

    blog_id = extract_blog_id_from_html(resp.text)

    if blog_id:
        result["shopify_blog_id"] = blog_id
        result["cms_url"]         = build_cms_url(STORE_NAME, blog_id)
        result["status"]          = "ok"
    else:
        snap_path = f"_snapshot_blog_{int(time.time())}.html"
        try:
            pathlib.Path(snap_path).write_text(resp.text, encoding="utf-8")
            result["status"] = f"id not found (snapshot: {snap_path})"
        except Exception:
            result["status"] = "id not found"

    return result

# ── CSV Helpers ──────────────────────────────────────────────
CSV_COLUMNS = ["url", "blog_slug", "shopify_blog_id", "cms_url", "status"]

def init_output_csv(output_path):
    pd.DataFrame(columns=CSV_COLUMNS).to_csv(output_path, index=False, encoding="utf-8")
    log("SAVE", f"Output CSV diinisiasi: {output_path}")

def append_to_csv(output_path, row):
    pd.DataFrame([row])[CSV_COLUMNS].to_csv(
        output_path, mode="a", header=False, index=False, encoding="utf-8"
    )

# ── Entry Point ──────────────────────────────────────────────
def main():
    try:
        with open(INPUT_FILE, "r", encoding="utf-8") as f:
            urls = [ln.strip() for ln in f if ln.strip()]
    except FileNotFoundError:
        log("ERROR", f"File tidak ditemukan: {INPUT_FILE}")
        return

    if not urls:
        log("WARN", "Tidak ada URL di file input.")
        return

    total = len(urls)
    log("INFO", f"Store    : {STORE_NAME}")
    log("INFO", f"Ditemukan {total} URL blog archive")
    print("=" * 60)

    init_output_csv(OUTPUT_FILE)

    success = 0
    failed  = 0
    skipped = 0

    for i, url in enumerate(urls, start=1):
        print(f"\n-- [{i}/{total}] ------------------------------------------")
        log("INFO",  f"URL      : {url}")

        result = fetch_blog_id(url)

        if result["status"] == "ok":
            log("OK",   f"Slug     : {result['blog_slug']}")
            log("OK",   f"Blog ID  : {result['shopify_blog_id']}")
            log("LINK", f"CMS URL  : {result['cms_url']}")
            success += 1
        elif result["status"] and result["status"].startswith("skipped"):
            log("SKIP", f"Reason   : {result['status']}")
            skipped += 1
        else:
            log("WARN", f"Status   : {result['status']}")
            failed += 1

        append_to_csv(OUTPUT_FILE, result)
        log("SAVE", f"Baris disimpan ke {OUTPUT_FILE}")

        if i < total:
            log("DELAY", f"Menunggu {DELAY_SECONDS} detik...")
            time.sleep(DELAY_SECONDS)

    print("\n" + "=" * 60)
    log("DONE", f"Selesai! Total: {total} | OK: {success} | Gagal: {failed} | Skip: {skipped}")
    log("SAVE", f"Hasil tersimpan di: {OUTPUT_FILE}")

main()

[13:32:20] i  [INFO] Store    : sme-brother
[13:32:20] i  [INFO] Ditemukan 6 URL blog archive
[13:32:20] S  [SAVE] Output CSV diinisiasi: shopify_blog_ids.csv

-- [1/6] ------------------------------------------
[13:32:20] i  [INFO] URL      : https://smebrother.com/blogs/貸款
[13:32:21] OK [OK] Slug     : 貸款
[13:32:21] OK [OK] Blog ID  : 87627038919
[13:32:21] @  [LINK] CMS URL  : https://admin.shopify.com/store/sme-brother/content/blogs/87627038919
[13:32:21] S  [SAVE] Baris disimpan ke shopify_blog_ids.csv
[13:32:21] ~  [DELAY] Menunggu 5 detik...

-- [2/6] ------------------------------------------
[13:32:26] i  [INFO] URL      : https://smebrother.com/blogs/審計-case-study
[13:32:26] OK [OK] Slug     : 審計-case-study
[13:32:26] OK [OK] Blog ID  : 91595112647
[13:32:26] @  [LINK] CMS URL  : https://admin.shopify.com/store/sme-brother/content/blogs/91595112647
[13:32:26] S  [SAVE] Baris disimpan ke shopify_blog_ids.csv
[13:32:26] ~  [DELAY] Menunggu 5 detik...

-- [3/6] -----------------

In [ ]:
from google.colab import files
import os

# Hardcoded agar tidak bergantung pada variabel dari cell lain
OUTPUT_FILE = "shopify_blog_ids.csv"

if os.path.exists(OUTPUT_FILE):
    print(f"Mendownload {OUTPUT_FILE}...")
    files.download(OUTPUT_FILE)
else:
    print("File output belum tersedia. Jalankan cell scraper terlebih dahulu.")

Mendownload shopify_blog_ids.csv...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import os

# Hardcoded agar tidak bergantung pada variabel dari cell lain
OUTPUT_FILE = "shopify_blog_ids.csv"

if os.path.exists(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE)
    print(f"Total rows  : {len(df)}")
    print(f"Berhasil    : {(df['status'] == 'ok').sum()}")
    print(f"Gagal       : {df['status'].str.startswith('error').sum()}")
    print(f"Skip        : {df['status'].str.startswith('skipped').sum()}")
    print()
    display(df)
else:
    print("File output belum tersedia.")

Total rows  : 6
Berhasil    : 6
Gagal       : 0
Skip        : 0



,url,blog_slug,shopify_blog_id,cms_url,status
0,https://smebrother.com/blogs/貸款,貸款,87627038919,https://admin.shopify.com/store/sme-brother/co...,ok
1,https://smebrother.com/blogs/審計-case-study,審計-case-study,91595112647,https://admin.shopify.com/store/sme-brother/co...,ok
2,https://smebrother.com/blogs/銀行開戶,銀行開戶,87648501959,https://admin.shopify.com/store/sme-brother/co...,ok
3,https://smebrother.com/blogs/跨境遺產-case-study,跨境遺產-case-study,91595210951,https://admin.shopify.com/store/sme-brother/co...,ok
4,https://smebrother.com/blogs/移民,移民,87640113351,https://admin.shopify.com/store/sme-brother/co...,ok
5,https://smebrother.com/blogs/驗貨,驗貨,87649550535,https://admin.shopify.com/store/sme-brother/co...,ok
